# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of Analysis & Time Window

* **Grain (One Row):**  
  Represents one unique content item (`content_hash_id`) per client (`client_hash_id`), aggregated over a 30 day period (2026-03-1 till 2026-03-31).

In [1]:
%pip -q install duckdb huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Verify uniqueness at the stated grain (client_hash_id x content_hash_id for March 2026)
grain_check_query = """
WITH march_aggregated AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        ANY_VALUE(gsc_avg_position) AS avg_position,
        COUNT(DISTINCT report_date) AS active_days
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
)
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    COUNT(*) - COUNT(DISTINCT content_hash_id) AS duplicate_check
FROM march_aggregated;
"""

# Ensure HF secret or setup is initialized prior if connecting remotely
grain_results = con.execute(grain_check_query).df()
print(grain_results)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_content_items  duplicate_check
0      331437                331437                0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.